#### Instructions

In the lesson, you used a subset of the pumpkin data. Now, go back to the original data and try to use all of it, cleaned and standardized, to build a Logistic Regression model.

In [31]:
import pandas as pd


original_df = pd.read_csv('../data/US-pumpkins.csv')

original_df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1757 entries, 0 to 1756
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   City Name        1757 non-null   object 
 1   Type             45 non-null     object 
 2   Package          1757 non-null   object 
 3   Variety          1752 non-null   object 
 4   Sub Variety      296 non-null    object 
 5   Grade            0 non-null      float64
 6   Date             1757 non-null   object 
 7   Low Price        1757 non-null   float64
 8   High Price       1757 non-null   float64
 9   Mostly Low       1654 non-null   float64
 10  Mostly High      1654 non-null   float64
 11  Origin           1754 non-null   object 
 12  Origin District  131 non-null    object 
 13  Item Size        1478 non-null   object 
 14  Color            1141 non-null   object 
 15  Environment      0 non-null      float64
 16  Unit of Sale     162 non-null    object 
 17  Quality       

In [32]:
# Columns that have 0 non-null values or unnamed
columns_to_drop = [
    'Grade',
    'Environment',
    'Quality',
    'Condition',
    'Appearance',
    'Storage',
    'Crop',
    'Trans Mode',
    'Unnamed: 24',
    'Unnamed: 25',
]

cleaned_df = original_df.drop(columns=columns_to_drop)

cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1757 entries, 0 to 1756
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   City Name        1757 non-null   object 
 1   Type             45 non-null     object 
 2   Package          1757 non-null   object 
 3   Variety          1752 non-null   object 
 4   Sub Variety      296 non-null    object 
 5   Date             1757 non-null   object 
 6   Low Price        1757 non-null   float64
 7   High Price       1757 non-null   float64
 8   Mostly Low       1654 non-null   float64
 9   Mostly High      1654 non-null   float64
 10  Origin           1754 non-null   object 
 11  Origin District  131 non-null    object 
 12  Item Size        1478 non-null   object 
 13  Color            1141 non-null   object 
 14  Unit of Sale     162 non-null    object 
 15  Repack           1757 non-null   object 
dtypes: float64(4), object(12)
memory usage: 219.8+ KB


Question: Can I classify what tyep of pumpkin pie it is?

In [35]:
features = cleaned_df.loc[:,['Variety', 'Color', 'Item Size', 'Origin', 'Repack']]
features = features.dropna()



# Encode the categorical features using one-hot encoding and ordinal encoding
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer 

one_hot_encoder = OneHotEncoder(sparse_output=False) 
ordinal_encoder = OrdinalEncoder(
    categories= [['sml', 'med', 'med-lge', 'lge', 'xlge', 'jbo', 'exjbo']]
) 


column_transformer = ColumnTransformer(
    transformers=[
        ('onehot', one_hot_encoder, ['Color', 'Repack', 'Origin']),
        ('ordinal', ordinal_encoder, ['Item Size']),
    ]
)
column_transformer.set_output(transform='pandas')
encoded_features = column_transformer.fit_transform(features)

# Encode the label
label_encoder = LabelEncoder()
encoded_label = label_encoder.fit_transform(features['Variety'])

# Combine the encoded features and label into a single DataFrame
encoded_data = encoded_features.assign(Variety=encoded_label)

In [46]:
# Train the logistic regression model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

x = encoded_data.drop(columns=['Variety'])
y = encoded_data['Variety']

print(x.shape)
print(y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=0
)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluate the model
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

print("Classification Report:")
print(classification_report(y_test, y_pred))

(991, 22)
(991,)
Confusion Matrix:
[[ 2  0  0  0  2  0  0  1  1]
 [ 0  0  0  0  1  0  0  0  0]
 [ 0  0  2  0  4  0  0  0  0]
 [ 0  0  3  0  2  0  0  0  4]
 [ 0  0  0  0 86  0  0  1  9]
 [ 0  0  0  0  0  7  0  4  0]
 [ 0  0  0  0  2  0  0  0  0]
 [ 0  0  0  0  1  0  0 18  6]
 [ 0  0  0  0 27  0  0  4 12]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         6
           1       0.00      0.00      0.00         1
           2       0.40      0.33      0.36         6
           3       0.00      0.00      0.00         9
           4       0.69      0.90      0.78        96
           5       1.00      0.64      0.78        11
           6       0.00      0.00      0.00         2
           7       0.64      0.72      0.68        25
           8       0.38      0.28      0.32        43

    accuracy                           0.64       199
   macro avg       0.46      0.36      0.38       199
weighted avg       0.

/Users/don/Desktop/machine-learning/ml-for-beginners/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/don/Desktop/machine-learning/ml-for-beginners/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/don/Desktop/machine-learning/ml-for-beginners/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame